# M20c2 — UCI Air Quality (datetime-corrected reconstruction)

**Author:** Ildefons Magrans de Abril  
**Affiliation:** Universitat Politècnica de Catalunya - BarcelonaTech (UPC)

**Purpose.** Reconstruct the corrected Air Quality experiment with a one-hour target horizon.

**Provenance.** The original June 2026 M20c2 notebook binary is unavailable. This notebook is a transparent reconstruction from recovered settings and implementation lineage. It is a fresh protocol replication, not a recovery of the historical experiment.

**v18.3 execution repairs.** The official UCI CSV uses decimal commas and `HH.MM.SS` time strings, so parsing is explicit. The manuscript-native real-data safe-region tolerance `epsilon_abs=0.002` is also passed explicitly. Fresh outputs are written only to `results/reproduced/`; publication-facing historical values are not overwritten.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import tcr_core as tcr

REPRO = ROOT / "results" / "reproduced"
REPRO.mkdir(parents=True, exist_ok=True)
PUB_TABLE = ROOT / "manuscript" / "tables" / "table_real_world.csv"

In [2]:
DATA = ROOT / "data" / "AirQualityUCI.csv"
URL = "https://archive.ics.uci.edu/static/public/360/air%2Bquality.zip"
HORIZON = 1
N = 50
K = 13
RIDGE = 1e-4
SEED = 20260718
ABS_TOL = 0.002

pub = pd.read_csv(PUB_TABLE)
display(pub[pub["Dataset"] == "Air Quality"])
print("data file:", DATA)
print("present:", DATA.exists())
print("native v18.3 epsilon_abs:", ABS_TOL)

,Dataset,Rows/features,Target,Temperature support range,Temperature full-test best-safe gain,Temperature safe - shuffled gain,Temperature path indicators
1,Air Quality,9356/19,"future C6H6, h=1",45.55,0.01635,"+0.05312 [0.00695, 0.09534]",0.727 / 0.646


data file: /mnt/data/tcr_repair/repo/data/AirQualityUCI.csv
present: True
native v18.3 epsilon_abs: 0.002


## Datetime- and decimal-corrected preprocessing

In [3]:
def prepare_air_quality(path):
    # Official UCI CSV: semicolon separator, decimal comma, and HH.MM.SS time strings.
    df = pd.read_csv(path, sep=";", decimal=",")
    df = df.dropna(axis=1, how="all")
    dt = pd.to_datetime(
        df["Date"].astype(str) + " " + df["Time"].astype(str),
        format="%d/%m/%Y %H.%M.%S",
        errors="coerce",
    )

    value_cols = [c for c in df.columns if c not in ["Date", "Time"]]
    numeric = df[value_cols].apply(pd.to_numeric, errors="coerce").replace(-200, np.nan)
    numeric = numeric.interpolate(limit_direction="both")
    valid = dt.notna()
    dt = dt[valid].reset_index(drop=True)
    numeric = numeric.loc[valid].reset_index(drop=True)

    X = numeric.copy()
    X["hour_sin"] = np.sin(2*np.pi*dt.dt.hour/24)
    X["hour_cos"] = np.cos(2*np.pi*dt.dt.hour/24)
    X["dow_sin"] = np.sin(2*np.pi*dt.dt.dayofweek/7)
    X["dow_cos"] = np.cos(2*np.pi*dt.dt.dayofweek/7)
    X["month_sin"] = np.sin(2*np.pi*(dt.dt.month-1)/12)
    X["month_cos"] = np.cos(2*np.pi*(dt.dt.month-1)/12)
    y = numeric["C6H6(GT)"].shift(-HORIZON)

    X = X.iloc[:-HORIZON].reset_index(drop=True)
    y = y.iloc[:-HORIZON].reset_index(drop=True)
    assert len(X) == 9356 and X.shape[1] == 19, (len(X), X.shape)
    assert not X.isna().any().any() and not y.isna().any()
    return X.to_numpy(float), y.to_numpy(float)

## Fresh reconstruction execution

In [4]:
if not DATA.exists():
    raise FileNotFoundError(f"Place the official UCI file at {DATA}")

X, y = prepare_air_quality(DATA)
print("prepared:", X.shape, y.shape)

ntr, nv, nt = 1500, 600, 600
Xtr, Xv, Xt = X[:ntr], X[ntr:ntr+nv], X[ntr+nv:ntr+nv+nt]
ytr, yv, yt = y[:ntr], y[ntr:ntr+nv], y[ntr+nv:ntr+nv+nt]
mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-12
Xtr = (Xtr-mu)/sd
Xv = (Xv-mu)/sd
Xt = (Xt-mu)/sd

rows = []
for path in ["temperature", "gain", "leak", "sparsity"]:
    rows.append(tcr.evaluate_arrays(
        Xtr, ytr, Xv, yv, Xt, yt,
        "air_quality", 0, path, SEED, N, K, RIDGE,
        abs_tol=ABS_TOL,
    ))
rep = pd.DataFrame(rows)
rep.to_csv(REPRO / "m20c2_replication_summary.csv", index=False)
display(rep.round(6))

prepared: (9356, 19) (9356,)


,task,trial,path,default_idx,safe_low_idx,safe_high_idx,safe_width,safe_width_fraction,near_contained,exact_contained,matched_near_rate,matched_oracle_rate,safe_gain,full_gain,matched_best_gain,safe_minus_matched_gain,default_test_nrmse,safe_best_test_nrmse,full_best_test_nrmse,support_range,spectral_radius_range
0,air_quality,0,temperature,8,8,9,2,0.153846,1,1,0.250000,0.166667,0.000000,0.000000,-0.034541,0.034541,0.803974,0.803974,0.803974,45.452504,0.216851
1,air_quality,0,gain,7,6,10,5,0.384615,0,0,0.222222,0.111111,0.028956,0.046771,0.009445,0.019511,0.795253,0.766297,0.748481,0.000000,1.603189
2,air_quality,0,leak,12,10,12,3,0.230769,1,1,0.272727,0.090909,0.000000,0.000000,-0.059487,0.059487,0.803673,0.803673,0.803673,0.000000,0.000000
3,air_quality,0,sparsity,6,6,6,1,0.076923,1,1,0.076923,0.076923,0.000000,0.000000,-0.073645,0.073645,0.754722,0.754722,0.754722,48.000000,0.093995


## Publication-reference comparison

In [5]:
fresh = rep.loc[rep.path == "temperature"].iloc[0]
published = float(pub.loc[pub["Dataset"] == "Air Quality", "Temperature full-test best-safe gain"].iloc[0])
comparison = pd.DataFrame([{
    "study": "Air Quality",
    "published_v18_3_gain": published,
    "fresh_reconstruction_gain": float(fresh.safe_gain),
    "fresh_safe_width": f"{int(fresh.safe_width)}/{K}",
    "exact_point_estimate_reproduced": bool(np.isclose(float(fresh.safe_gain), published, atol=1e-8)),
}])
display(comparison)
print("Fresh reconstruction is provenance-separated and does not overwrite the publication table.")

,study,published_v18_3_gain,fresh_reconstruction_gain,fresh_safe_width,exact_point_estimate_reproduced
0,Air Quality,0.01635,0.0,2/13,False


Fresh reconstruction is provenance-separated and does not overwrite the publication table.
